# NeuroScan Fine-tune — SIMPLE (no Drive upload needed)

## Setup (once)
1. **Runtime → Change runtime type → GPU (T4) → Save**
2. Run **Cell 1** below
3. When asked, upload **2 files from your PC**:
   - `NeuroScan_Nepal/scripts/neuroscan_data.zip`
   - `NeuroScan_Nepal/models/cnn_baseline.pth`
4. Wait ~5–15 minutes for training
5. Download `cnn_baseline.pth` at the end

In [ ]:
# === RUN THIS ONE CELL ONLY ===
!pip install -q opencv-python-headless scikit-learn matplotlib

import torch
from google.colab import files
from pathlib import Path
import zipfile, urllib.request, shutil

assert torch.cuda.is_available(), 'ERROR: Enable GPU → Runtime → Change runtime type → GPU (T4)'
print('GPU OK:', torch.cuda.get_device_name(0))

WORK = Path('/content/neuroscan')
OUT = Path('/content/output')
WORK.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)

# 1) Upload dataset zip
print('\n=== STEP 1/3: Choose neuroscan_data.zip (from scripts folder on PC) ===')
up = files.upload()
if not up:
    raise FileNotFoundError('No file selected. Pick neuroscan_data.zip')
zip_name = next(iter(up))
zip_path = WORK / 'neuroscan_data.zip'
zip_path.write_bytes(up[zip_name])
print(f'Got {zip_name} ({len(up[zip_name])/1e6:.1f} MB)')

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(WORK)
DATA_ROOT = WORK if (WORK/'normal').exists() else WORK/'raw'
n = len(list((DATA_ROOT/'normal').rglob('*.jpg')))
a = len(list((DATA_ROOT/'abnormal').rglob('*.jpg')))
print(f'Images: normal={n}, abnormal={a}')
if n == 0 or a == 0:
    raise FileNotFoundError('Zip has no images. On PC run: scripts/prepare_colab_upload.ps1')

# 2) Upload pretrained model
print('\n=== STEP 2/3: Choose cnn_baseline.pth (from models folder on PC) ===')
up = files.upload()
if not up:
    raise FileNotFoundError('No file selected. Pick cnn_baseline.pth')
pth_name = next(iter(up))
PRETRAINED = OUT / 'cnn_baseline_pretrained.pth'
PRETRAINED.write_bytes(up[pth_name])
print(f'Got {pth_name} ({len(up[pth_name])/1e6:.1f} MB)')

# 3) Download training script from GitHub (no manual upload)
print('\n=== STEP 3/3: Download training script from GitHub ===')
SCRIPT_URL = 'https://raw.githubusercontent.com/pragya23189634-sys/neuroscannepal/main/notebooks/colab/neuroscan_colab_train.py'
script_path = Path('neuroscan_colab_train.py')
urllib.request.urlretrieve(SCRIPT_URL, script_path)
print('Script downloaded:', script_path.exists())

# 4) Fine-tune
print('\n=== TRAINING (fine-tune, ~5-15 min) ===')
!python neuroscan_colab_train.py --data-root {DATA_ROOT} --out-dir {OUT} --pretrained {PRETRAINED} --epochs 15 --batch-size 64 --lr 0.0001 --patience 5

# 5) Download results
print('\n=== DOWNLOAD RESULTS ===')
for f in OUT.iterdir():
    files.download(str(f))
    print('Downloaded:', f.name)
print('\nDone! Copy cnn_baseline.pth to NeuroScan_Nepal/models/ on your PC')